# Colorização de Imagens Usando U-Net

## Introdução

Neste laboratório, será abordado um problema de tradução de imagem para imagem (*Image-to-Image Translation*). O desafio proposto é generativo, na qual a rede neural deve inferir a informação de crominância (cor) ausente baseando-se estritamente na geometria e nas texturas apresentadas no canal de luninância (escala de cinza).

Para viabilizar essa reconstrução com boa fidelidade, a escolha da arquitetura é um fator importante. Um autocodificador padrão seria insuficiente, pois seu gargalo de compressão tende a descartar informações espaciais finas, resultando em saídas borradas. Por isso, será utilizado a U-Net. A principal vantagem estrutural da U-Net reside nas *skip connection* (conexões de atalho). Essas vias permitem que o fluxo de dados salte o gargalo da compressão, transpondo detalhes de alta frequência (bordas e contornos) diretamente do codificador para o decodificador. Isso garante que a imagem colorizada mantenha a nitidez estrutural da entrada original, resolvendo o problema de degradação espacial típico de redes convolucionais profundas [<a href="#ref1">1</a>].

<figure style="text-align: center;">
  <img src="assets/figura-2.png" alt="Figura 1" width="800">
</figure>

**Figura 1** – Diagrama arquitetural da U-Net, uma versão mais sofisticada de um autocodificador tradicional. [<a href="#ref1">1</a>]

## Preparação dos Dados

Para este laboratório, é utilizado o dataset "Landscapes", composto por aproximadamente 4.300 imagens de cenários naturais, incluindo montanhas, florestas, lagos e praias. A escolha deste conjunto de dados é devido a existência de cenas de imagens de natureza, na qual possuem uma forte correlação semântica entre textura e cor. Por exemplo, a textura fluida da água geralmente indica tons de azul, enquanto texturas fibrosas de vegetação indicam verde. Esses padrões facilitam o aprendizado da rede neural, de modo que ela associe padrões geométricos (luminância) a informações cromáticas (cores) com mais facilidade do que em conjuntos de imagens com objetos artificiais, onde a cor pode ser aleatória (como um carro, que pode ter qualquer cor independentemente de sua forma).

Ao carregar as imagens, note que foi definido`label_mode=None`. Isso reforça que o próprio dado é o nosso alvo: a entrada ($X$) será a luminância (escala de cinza) e o alvo ($Y$) será a imagem original RGB. É importante destacar a normalização dos pixels (dividindo por 255), pois redes neurais convergem melhor e mais estável quando os dados operam em uma escala pequena, entre 0 e 1, evitando explosões de gradientes.

In [ ]:
#!pip install scikit-image

In [ ]:
from tensorflow.keras import layers, models, callbacks
import matplotlib.pyplot as plt
from skimage import color
from pathlib import Path
import tensorflow as tf
import numpy as np
import zipfile
import os

In [ ]:
# Configuração para reprodutibilidade
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

In [ ]:
IMG_SIZE = 128
BATCH_SIZE = 32
EPOCHS = 25

In [ ]:
# baixe o dataset pelo google drive
%pip install -q gdown

import gdown

FILE_ID = "1luDUiWUP4RCNUOoBS1QtucIBc60LNmQt"
DATASET_ZIP = "assets/landscapes.zip"
DATASET_DIR = "dataset_landscapes"

gdown.download(
    id=FILE_ID,
    output=DATASET_ZIP
)

In [ ]:
def setup_dataset(zip_path, extract_to):
    # extraí os dados compactados a partir de um arquivo .zip.

    path = Path(extract_to)
    if not path.exists():
        print(f"Extraindo {zip_path}...")
        if os.path.exists(zip_path):
            with zipfile.ZipFile(zip_path, "r") as zip_ref:
                zip_ref.extractall(extract_to)
            print("Extração concluída.")
        else:
            print(f"{zip_path} não encontrado.")
    else:
        print(f"Diretório {extract_to} já existente.")

    return path
        
data_dir = setup_dataset(DATASET_ZIP, DATASET_DIR)

In [ ]:
def load_data_from_disk(data_path):
    # usa image_dataset_from_directory: o padrão para carregar pastas de imagens.
    # label_mode=None, pois não é uma tarefa de classificação, buscamos apenas as imagens.

    dataset = tf.keras.utils.image_dataset_from_directory(
        data_path, labels=None,
        label_mode=None, color_mode="rgb",
        batch_size=BATCH_SIZE, image_size=(IMG_SIZE, IMG_SIZE),
        shuffle=True, seed=SEED,
        interpolation="bilinear"
    )

    return dataset

full_ds = load_data_from_disk(data_dir)

In [ ]:
dataset_size = tf.data.experimental.cardinality(full_ds).numpy()

train_size = int(0.7 * dataset_size)
val_size = int(0.15 * dataset_size)
test_size = int(0.15 * dataset_size)

train_ds = full_ds.take(train_size)
val_ds = full_ds.skip(train_size).take(val_size)
test_ds = full_ds.skip(train_size + val_size)

In [ ]:
def process_input_target(image):
    # transforma a imagem crua nos pares (input, target)
    # input: escala de cinza (luminância) normalizada.
    # targetÇ RGB original normalizado.

    image = tf.cast(image, tf.float32) / 255.0
    grayscale = tf.image.rgb_to_grayscale(image)

    return grayscale, image

In [ ]:
train_ds = (train_ds.shuffle(1000).map(process_input_target, num_parallel_calls=tf.data.AUTOTUNE)
            .cache().prefetch(tf.data.AUTOTUNE))

val_ds = (val_ds.map(process_input_target, num_parallel_calls=tf.data.AUTOTUNE)
          .cache().prefetch(tf.data.AUTOTUNE))

test_ds = (test_ds.map(process_input_target, num_parallel_calls=tf.data.AUTOTUNE)
           .cache().prefetch(tf.data.AUTOTUNE))

print(f"Total de batches: {dataset_size}")
print(f"Treino: {tf.data.experimental.cardinality(train_ds).numpy()} batches.")
print(f"Validação: {tf.data.experimental.cardinality(val_ds).numpy()} batches.")
print(f"Teste: {tf.data.experimental.cardinality(test_ds).numpy()} batches.")

In [ ]:
def check_pipeline(dataset):
    # verificação do pipeline construído para carregar as imagens.

    gray_batch, rgb_batch = next(iter(dataset))
    plt.figure(figsize=(10, 5))
    for i in range(4):
        ax = plt.subplot(2, 4, i + 1)
        plt.imshow(tf.squeeze(gray_batch[i]), cmap="gray")
        plt.title("Entrada (P&B)")
        plt.axis("off")

        ax = plt.subplot(2, 4, i + 1 + 4)
        plt.imshow(rgb_batch[i])
        plt.title("Imagem Cor (RGB)")
        plt.axis("off")
    plt.show()

check_pipeline(test_ds)

## Modelagem da Arquitetura

Na U-Net, o ramo descendente (codificador) atua como um extrator de características, comprimindo a espacialidade para captuar o contexto global da cena. O ramos ascendente (decodificador) realiza a expansão. O diferencial críitico reside nas operações de `Concatenate` (*skip connections*). Essas conexões preservam os mapas de características de alta frequência do codificador e os fundem com a saída do decodificador. Isso permite que a rede capture detalhes finos e bordas nítidas que seriam irremediavelmente perdidos em um autocodificador convencional puramente sequencial. Vale a pena destacar que a qualidade da nitidez não vem do decodificador aprender a fazer as bordas do zero, mas sim da concatenação de mapas de características que armazenam bem esses detalhes preservados no codificador.

O treinamento é conduzido utilizando o erro quadrático médio (MSE). Embora eficiente computacionalmente, o MSE apresenta limitações severas para tarefas generativas multimodais como a colorização. Por tender à média estatística das distribuições de cor possíveis, o modelo frequentemente converge para tons dessaturados ou acizentados, uma solução conservadora que minimiza a penalidade numérica, mas sacrifica a vivacidade perceptual. O uso de *callbacks* como `ReduceLROnPlateau` e `EarlyStopping` mitiga o *overfitting*, ajustando dinamicamente a taxa de aprendizado e interrompendo o processo assim que a perda de validação estagna.

In [ ]:
def build_unet():
    # modelagem da arquitetura u-net
    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 1))

    # codificador
    c1 = layers.Conv2D(64, 3, activation="relu", padding="same", strides=2)(inputs) # 64x64
    c1_out = layers.Conv2D(64, 3, activation="relu", padding="same")(c1) # Skip

    c2 = layers.Conv2D(128, 3, activation="relu", padding="same", strides=2)(c1_out) # 32x32
    c2_out = layers.Conv2D(128, 3, activation="relu", padding="same")(c2) # Skip

    c3 = layers.Conv2D(256, 3, activation="relu", padding="same", strides=2)(c2_out) # 16x16
    
    # gargalo (espaço latente)
    bottleneck = layers.Conv2D(512, 3, activation="relu", padding="same")(c3)

    # decodificador
    u1 = layers.Conv2DTranspose(256, 3, strides=2, padding="same")(bottleneck)
    u1 = layers.Concatenate()([u1, c2_out])
    c4 = layers.Conv2D(256, 3, activation="relu", padding="same")(u1)
    
    u2 = layers.Conv2DTranspose(128, 3, strides=2, padding="same")(c4)
    u2 = layers.Concatenate()([u2, c1_out])
    c5 = layers.Conv2D(128, 3, activation="relu", padding="same")(u2)

    u3 = layers.Conv2DTranspose(64, 3, strides=2, padding="same")(c5)
    c6 = layers.Conv2D(64, 3, activation="relu", padding="same")(u3)

    outputs = layers.Conv2D(3, 3, activation="sigmoid", padding="same")(c6)

    return models.Model(inputs, outputs, name="Landscape_Colorizer")

In [ ]:
model = build_unet()
model.compile(optimizer="adam", loss="mse")

<div style="
    background-color:#f9f9f9;
    border-left:6px solid #4A90E2;
    padding:15px;
    border-radius:8px;
    margin:10px 0;
">
  <b style="color:#2c3e50;"> Pergunta 1:</b><br>
  O texto menciona que um autocodificador padrão resultaria em saídas borradas devido ao gargalo de compressão. Pesquisa e explique tecnicamente como as Skip Connections (conexões de atalho) da U-Net resolvem esse problema de perda de detalhes espaciais na reconstrução da imagem.
</div>

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[
        callbacks.EarlyStopping(patience=5, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(factor=0.5, patience=2)
    ]
)

# Salva o modelo treinado

# MODEL_DIR = "saved_models"
# MODEL_NAME = "landscape_colorizer_unet.keras"
# MODEL_PATH = os.path.join(MODEL_DIR, MODEL_NAME)

# os.makedirs(MODEL_DIR, exist_ok=True)
# model.save(MODEL_PATH)

In [ ]:
# # baixe os pesos treinados do modelo se não quiser treiná-lo

# %pip install -q gdown

# import gdown

# FILE_ID = "1J_HqslWb3o5uW_RLYhqcmZ6ZflqntWU2"
# DATASET = MODEL_PATH

# gdown.download(
#     id=FILE_ID,
#     output=DATASET
# )

# model = tf.keras.models.load_model(MODEL_PATH)

In [ ]:
def show_predictions(dataset, model, num=5):
    # visualiza as predições do modelo em algumas amostras de teste.

    gray, rgb = next(iter(dataset))
    preds = model.predict(gray)

    plt.figure(figsize=(12, num*3))
    for i in range(num):
        plt.subplot(num, 3, i*3 + 1)
        plt.imshow(tf.squeeze(gray[i]), cmap="gray")
        plt.title("Entrada (Escala de Cinza)")
        plt.axis("off")

        plt.subplot(num, 3, i*3 + 2)
        plt.imshow(preds[i])
        plt.title("U-Net")
        plt.axis("off")

        plt.subplot(num, 3, i*3 + 3)
        plt.imshow(rgb[i])
        plt.title("Original")
        plt.axis("off")
    plt.tight_layout()
    plt.show()

show_predictions(test_ds, model)

## Análise dos Resultados

A avaliação qualitativa reve que o modelo RGB possui dificuldades em dissociar luminância de cromância, visto que essas informações são correlacionadas nos três canais vermelho, verde e azul. Isso impõe uma redundância desnecessária ao treinamento. Uma otimização interessante para esse *pipeline* seria a transição para o espaço de cor CIELAB. Neste espaço, a luminância ($L$) é totalmente desacoplada da informação da cor ($a, b$). Como o canal $L$ já é fornecido como entrada, a rede neural poderia focar exclusivamente na predição dos canais ($a$, $b$), reduzindo a dimensionalidade do problema e eliminando a necessidade de reaprender a estrutura de iluminação da cena.

## Modificando o Espaço de Cor para CIELAB

Até o momento, a intuição natural seria trabalhar no espaço RGB (Vermelho, Verde e Azul), pois é como as telas exibem imagens. No entanto, para tarefas de visão computacional no ramo generativo, em específico a colorização, o RGB apresenta uma alta correlação entre canais. No RGB, a informação de brilho (luz e sombra) está espalhada pelos três canais simultaneamente. Se quisermos escurecer um pixel vermelho, precisamos alterar os valores de R, G e B. Isso obriga a rede neural a aprender duas tarefas concorrentes, que são reconstruir a estrutura da cena (iluminação) e escolher a cromaticidade (cor).

Para otimizar o modelo, é adotado o espaço de cor CIELAB. Este espaço opera um desacoplamento que pode ser explicado da seguinte forma:

* **Canal L (Lightness)**: Representa puramente a luminosidade, variando de 0 (preto) a 100 (branco). Este canal é idêntico à imagem de entrada em escala de cinza.
* **Canal $a$ (Verde-Vermelho)**: Um eixo onde valores negativos tendem ao verde e positivos ao vermelho.
* **Canal $b$ (Azul-Amarelo)**: Um eixo onde valores negativos tendem ao azul e positivos ao amarelo.

Como a entrada da rede é a imagem em escala de cinza, o canal $L$ já está definido, não é necessário que a rede o preveja. A tarefa da U-Net fica simplificada, na qual ela só precisa prever os canais ($a$, $b$). Na prática, isso significa que não estamos pedindo para a colorização acontecer do zero, mas sim decidir quais cores ($a$, $b$) preenchem os espaços. Isso resulta em um treinamento mais rápido e imagens com nitidez estrutural, pois a resolução original do canal $L$ é preservado na saída final.

<figure style="text-align: center;">
  <img src="assets/Figura-3.png" alt="Figura 3" width="800">
  <figcaption><em><b>Figura 3</b> – Espaço de cores CIELAB. [<a href="#ref3">3</a>]</em></figcaption>
</figure>

<div style="
    background-color:#f9f9f9;
    border-left:6px solid #4A90E2;
    padding:15px;
    border-radius:8px;
    margin:10px 0;
">
  <b style="color:#2c3e50;"> Pergunta 2:</b><br>
 No primeiro modelo, a rede precisava predizer 3 canais (RGB). No modelo CIELAB, a entrada é o canal L e a saída são os canais (a, b). Por que essa mudança de espaço de cor facilita o processo de convergência da rede neural e garante, estruturalmente, uma imagem final mais nítida que a abordagem RGB?
</div>



In [ ]:
# espaço de luminosidade de (0 a 100)
L_RANGE = 100.0
# canais de cor vão de -128 até 127
AB_RANGE = 128.0

In [ ]:
def rgb_to_lab_norm(image_rgb):
    # converte uma imagem RGB para CIELAB e aplica normalização.

    lab_image = color.rgb2lab(image_rgb)

    l_channel = lab_image[:, :, 0]
    ab_channels = lab_image[:, :, 1:]

    l_norm = l_channel / L_RANGE
    ab_norm = ab_channels / AB_RANGE
    
    h, w = l_norm.shape
    l_norm = l_norm.reshape((h, w, 1))

    return l_norm.astype(np.float32), ab_norm.astype(np.float32)

In [ ]:
def tf_process_lab(image_path):
    # ajusta as imagens como tensores no padrão CIELAB.

    img = tf.io.read_file(image_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32) / 255.0

    l_norm, ab_norm = tf.numpy_function(
        func=rgb_to_lab_norm,
        inp=[img],
        Tout=[tf.float32, tf.float32]
    )

    l_norm.set_shape([IMG_SIZE, IMG_SIZE, 1])
    ab_norm.set_shape([IMG_SIZE, IMG_SIZE, 2])

    return l_norm, ab_norm

In [ ]:
list_ds = tf.data.Dataset.list_files(str(DATASET_DIR + "/*.jpg"), shuffle=True, seed=SEED)

In [ ]:
val_size = int(len(list_ds) * 0.15)
test_size = int(len(list_ds) * 0.15)
train_size = int(len(list_ds)) - val_size - test_size

train_ds_lab = list_ds.take(train_size)
val_ds_lab = list_ds.skip(train_size).take(val_size)
test_ds_lab = list_ds.skip(train_size + val_size)

In [ ]:
def configure_for_performance(ds):
    # aplicando melhorias para acelerar a performance de treinamento.
    
    ds = ds.map(tf_process_lab, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(buffer_size=tf.data.AUTOTUNE)
    
    return ds

In [ ]:
BATCH_SIZE = 16

train_lab_ready = configure_for_performance(train_ds_lab)
val_lab_ready = configure_for_performance(val_ds_lab)
test_lab_ready = configure_for_performance(test_ds_lab)

In [ ]:
for l_batch, ab_batch in train_lab_ready.take(1):
    print(f"Dimensão de L (Entrada): {l_batch.shape}")   # (32, 128, 128, 1)
    print(f"Dimensão de AB (Alvo): {ab_batch.shape}") # (32, 128, 128, 2)

In [ ]:
def build_unet_lab():
    # modelo arquitetural da u-net no espaço de cor CIELAB.

    # entrada deve ser apenas o canal L 
    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 1)) 

    # codificador
    c1 = layers.Conv2D(64, 3, activation="relu", padding="same", strides=2)(inputs)
    c1_out = layers.Conv2D(64, 3, activation="relu", padding="same")(c1)

    c2 = layers.Conv2D(128, 3, activation="relu", padding="same", strides=2)(c1_out)
    c2_out = layers.Conv2D(128, 3, activation="relu", padding="same")(c2)

    c3 = layers.Conv2D(256, 3, activation="relu", padding="same", strides=2)(c2_out)

    # gargalo (espaço latente)
    bottleneck = layers.Conv2D(512, 3, activation="relu", padding="same")(c3)

    # Decodificador (Usando UpSampling para evitar "checkerboard artifacts")
    u1 = layers.UpSampling2D((2, 2))(bottleneck)
    u1 = layers.Concatenate()([u1, c2_out])
    c4 = layers.Conv2D(256, 3, activation="relu", padding="same")(u1)

    u2 = layers.UpSampling2D((2, 2))(c4)
    u2 = layers.Concatenate()([u2, c1_out])
    c5 = layers.Conv2D(128, 3, activation="relu", padding="same")(u2)

    u3 = layers.UpSampling2D((2, 2))(c5)
    c6 = layers.Conv2D(64, 3, activation="relu", padding="same")(u3)

    # saída: dois canais (a, b) e uma função de ativação tangente (-1 to 1 range)
    outputs = layers.Conv2D(2, 3, activation="tanh", padding="same")(c6)

    return models.Model(inputs, outputs, name="Landscape_Colorizer_CIELAB")

In [ ]:
model_lab = build_unet_lab()
model_lab.compile(optimizer="adam", loss="mse")
model_lab.summary()

In [ ]:
checkpoint_cb = callbacks.ModelCheckpoint(
    filepath="saved_models/landscape_colorizer_unet_cielab.keras",
    save_best_only=True,
    monitor="val_loss",
    mode="min",
    verbose=1
)

early_stop_cb = callbacks.EarlyStopping(
    patience=10,
    restore_best_weights=True,
    monitor="val_loss",
    verbose=1
)

reduce_lr_cb = callbacks.ReduceLROnPlateau(
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    monitor="val_loss",
    verbose=1
)

callbacks_list = [checkpoint_cb, early_stop_cb, reduce_lr_cb]

In [ ]:
history_lab = model_lab.fit(
    train_lab_ready,
    validation_data=val_lab_ready,
    epochs=EPOCHS,
    callbacks=callbacks_list
)

In [ ]:
# # baixe os pesos treinados do modelo se não quiser treiná-lo

# %pip install -q gdown

# import gdown

# FILE_ID = "1Fwn59x33gFVMFCYT8f9eS3bV2ajxWypO"
# DATASET = "saved_models/landscape_colorizer_unet_cielab.keras"

# gdown.download(
#     id=FILE_ID,
#     output=DATASET
# )

# model_lab = tf.keras.models.load_model(DATASET)

In [ ]:
def lab_to_rgb_denorm(l_norm, ab_norm):
    # reconstroí as imagens RGB a partir das imagens CIELAB.
    
    # desnormalização
    l_channel = l_norm * L_RANGE
    ab_channels = ab_norm * AB_RANGE

    # vetoriza a informação L e (a,b)
    lab_image = np.dstack((l_channel, ab_channels))

    # CIELAB -> RGB 
    rgb_image = color.lab2rgb(lab_image)
    
    return rgb_image

In [ ]:
def show_lab_predictions(dataset, model, num=5):
    # visualiza as predições do modelo em algumas amostras de teste
    
    l_batch, ab_batch = next(iter(dataset))
    preds_ab = model.predict(l_batch)

    plt.figure(figsize=(12, num * 3))

    for i in range(num):
        curr_l = l_batch[i].numpy() # (128, 128, 1)
        curr_ab_true = ab_batch[i].numpy()
        curr_ab_pred = preds_ab[i]

        img_original = lab_to_rgb_denorm(curr_l[:,:,0], curr_ab_true)
        img_pred = lab_to_rgb_denorm(curr_l[:,:,0], curr_ab_pred)

        plt.subplot(num, 3, i*3 + 1)
        plt.imshow(curr_l[:,:,0], cmap="gray")
        plt.title("Entrada (L)")
        plt.axis("off")

        plt.subplot(num, 3, i*3 + 2)
        plt.imshow(img_pred)
        plt.title("Predição (CIELAB -> RGB)")
        plt.axis("off")

        plt.subplot(num, 3, i*3 + 3)
        plt.imshow(img_original)
        plt.title("Original")
        plt.axis("off")

    plt.tight_layout()
    plt.show()

show_lab_predictions(test_lab_ready, model_lab)

## Análise dos Resultados

Ao analisarmos o lote de predições gerado aleatoriamente, podemos identificar comportamentos sistemáticos que independem da imagem específica. O modelo exibe características clássicas de algoritmos de colorização baseados em regressão (MSE). Alguns detalhes sobre esses resultados podem ser discutidos pontualmente abaixos.

* O uso da função de perda MSE melhora a qualidade visual das imagens. Se o modelo não tem certeza se um objeto é vermelho vibrante ou azul vibrante, a "média matemática" entre essas cores tende ao cinza ou marrom. Para minimizar a penalidade do erro, a rede prefre prever um tom pastel conservador do que arriscar uma cor forte e errar.
* A U-Net aprende a associar texturas locais a cores. A textura da nuvem quase sempre implica branco/cinza. A textura da folhagem quase sempre implica verde. Porém, objetos manufaturados (como carros, roupas ou paredes) não possuem uma correlação fixa entre textura e cor (um carro pode ter qualquer cor). Nesses casos, o modelo frequentemente falha ou deixa o objeto acizentado.
* Apesar das cores às vezes serem imprecisas, notem que a imagem final é nítida, com todas as bordas e sombras. Isso é mérito do espaço de cor CIELAB. Como nós fornecemos o canal $L$ original e intocado na reconstrução final, toda a informação de detalhe, sombra e contraste é preservada. A rede neural está apenas colorindo uma camada de cor ($a$, $b$) sobre uma base preto e branco. Se usássemos RGB puro, veríamos borrões nas bordas dos objetos.

Desse modo, neste laboratório foi possível perceber que é plausível recuperar informações de cor usando redes convolucionais, mas também expõe que a precisão matemática usando o MSE não garante uma colorização muito fidedigna a versão da imagem original. Para superar o aspecto "sépia" e gerar cores vibrantes, o estado da arte evoluiu para o uso de GANs (Redes Adversariais Generativas) [<a href="#ref2">2</a>], onde um segundo modelo (chamado de discriminador) força a U-Net a criar cores que enganam o olho humano, e não apenas minimiza uma equação. Em um próximo laboratório esse mesmo problema será retomado usando arquiteturas do tipo GANs.

<div style="
    background-color:#f9f9f9;
    border-left:6px solid #4A90E2;
    padding:15px;
    border-radius:8px;
    margin:10px 0;
">
  <b style="color:#2c3e50;"> Pergunta 3:</b><br>
 Ao analisar as imagens geradas, nota-se que muitas cores tendem a tons pastéis ou acinzentados (efeito sépia), evitando cores vibrantes e arriscadas. Relacione esse comportamento com a natureza matemática da função de perda Mean Squared Error (MSE) em cenários muito coloridos (onde um mesmo objeto pode ter várias cores válidas).
</div>

## Referências

<p id="ref1">[1] Ronneberger, O., Fischer, P., & Brox, T. (2015, October). U-net: Convolutional networks for biomedical image segmentation. In International Conference on Medical image computing and computer-assisted intervention (pp. 234-241). Cham: Springer international publishing.
<p id="ref2">[2] Saxena, D., & Cao, J. (2021). Generative adversarial networks (GANs) challenges, solutions, and future directions. ACM Computing Surveys (CSUR), 54(3), 1-42.
<p id="ref3">[3] Zhang, X., & Wandell, B. A. (1996, May). A spatial extension of CIELAB for digital color image reproduction. In SID international symposium digest of technical papers (Vol. 27, pp. 731-734). Society for Information Display.